# Encrypted Quipu Test 53 — ECIES N-of-N multisig sender (multiman 2-of-2)

**Sender (payer + ECIES author)**: the `multiman` 2-of-2 multisig of apocrypha + key1 at address `A3Shjwjs…`. Tx signing uses `CadenaMultiAtom`; ECIES author identity is the aggregate pubkey.

## How this protocol works

Same hybrid ECIES scheme as notebook 52, with one structural change: the **sender's identity is an aggregate** of two component keys — `mi_prv` (apocrypha's key) and `key1_prv` — forming the on-chain 2-of-2 multisig `multiman` at address `A3Shjwjs…`.

**Two separate uses of multisig in this notebook:**

**A. Transaction-signing side (on chain).** Every Dogecoin tx that spends from `multiman` is a P2SH 2-of-2 multisig spend — its scriptSig contains the redeem script (with both component pubkeys) plus both signatures. Standard Bitcoin/Dogecoin multisig. The orchestrator is `CadenaMultiAtom`, which precomputes the entire strand chain and signs each tx with both cosigners.

**B. ECIES-encryption side (mathematical).** The author identity for ECDH is the **aggregate pubkey**:
```
aggregate_pubkey  = pub_mi + pub_key1                          [point addition]
aggregate_privkey = (priv_mi + priv_key1) mod n                [scalar addition mod curve order]
```
Algebraic identity: `aggregate_privkey · G == aggregate_pubkey`. Anyone reading the chain can derive the aggregate pubkey by reading the redeem script and summing the component pubkeys. But the aggregate **privkey** requires both component privkey holders to cooperate (2-of-2).

**Why N-of-N (not M-of-N)?** Because to reconstruct the aggregate privkey, you need ALL N component privkeys. M-of-N threshold doesn't work for aggregate-key ECIES decryption.

**Three decode paths in this notebook:**

| envelope | recipient identity | how to decrypt |
|---|---|---|
| 0 | sender self — 2-of-2 aggregate (multiman) | reconstruct (mi + key1) aggregate privkey |
| 1 | single-key (`pub_test1`) | `priv_test1` + sender aggregate pubkey |
| 2 | DIFFERENT multikey aggregate — `test_multisig3` (3-of-3 of test1+test2+test3) | reconstruct (test1+test2+test3) aggregate privkey + sender aggregate pubkey |

**Fee considerations.** Multisig txs are much bigger than single-key:
- Single-key strand knot: ~250 B
- Multisig strand knot: ~400 B
- Multisig join tx with 5 inputs: ~1700 B

This notebook uses `TIP_MULTI = 0.10 DOGE per knot` for the multisig strand chains (matched to the 0.2 DOGE/KB target rate for a ~460-byte 2-of-2 knot), plus `scaled_fee()` for root and join at 0.2 DOGE/KB. The single-key strand TIP of 0.05 DOGE per knot already encodes the same 0.2 DOGE/KB rate for a ~250-byte knot — so all fees in the protocol are at one consistent rate. Wait cells after each broadcast confirm before proceeding.

**Payer**: `multiman` 2-of-2 multisig at A3Sh (mi + key1).

## Setup

In [9]:
import warnings
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL')

import os, sys, json, time, secrets
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

import cryptos
import colegio_tools as ct
from colegio_tools import _txid_of_serial
from text import build_text_quipu, read_text_quipu
from encrypted import (build_aes_quipu, build_ecies_quipu, build_keydrop_quipu,
                       read_encrypted_quipu, aggregate_privkey, aggregate_pubkey,
                       TONE_ORDINARY, TONE_AFFECTION, TONE_REVERENCE)
from coincurve import PrivateKey as CCPriv, PublicKey as CCPub

doge = cryptos.Doge()
TIP_SINGLE = 5_000_000     # 0.05 DOGE per knot for single-key strand txs
TIP_MULTI  = 10_000_000    # 0.10 DOGE per knot for multisig strand txs (size-matched to 0.2 DOGE/KB)
FEE_PER_KB = 20_000_000   # 0.2 DOGE/KB target for root + join txs

def scaled_fee(draft_hex_str, floor_sat):
    """Compute fee from drafted signed-tx size at FEE_PER_KB, floor at given TIP."""
    size_bytes = len(draft_hex_str) // 2
    return max(floor_sat, (size_bytes * FEE_PER_KB) // 1000)

In [10]:
LLAVES = os.path.abspath('../../cinv/llaves')
INSCRIPTIONS_READY = os.path.join(REPO, 'inscriptions_ready')
os.makedirs(INSCRIPTIONS_READY, exist_ok=True)

def load_priv(name, password=''):
    enc = open(os.path.join(LLAVES, f'{name}_prv.enc'), 'rb').read()
    return ct.import_privKey_from_bytes(enc, password)

priv_apo  = load_priv('mi')
priv_key1 = load_priv('key1')
priv_test1 = load_priv('test1')
priv_test2 = load_priv('test2')
priv_test3 = load_priv('test3')

addr_apo = doge.privtoaddr(priv_apo.to_hex()[2:])

multiman = json.load(open(os.path.join(LLAVES, 'multiman_multisig.json')))
addr_multiman = multiman['address']
redeem_multiman_hex = multiman['redeem_script_hex']

def cc_priv(p): return CCPriv(bytes.fromhex(p.to_hex()[2:]))
def cc_pub_from_priv(p): return cc_priv(p).public_key

pub_apo   = cc_pub_from_priv(priv_apo)
pub_key1  = cc_pub_from_priv(priv_key1)
pub_test1 = cc_pub_from_priv(priv_test1)
pub_test2 = cc_pub_from_priv(priv_test2)
pub_test3 = cc_pub_from_priv(priv_test3)

print(f'apocrypha payer:        {addr_apo}')
print(f'multiman 2-of-2 payer:  {addr_multiman}  (mi + key1)')

apocrypha payer:        D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX
multiman 2-of-2 payer:  A3ShjwjsAE4ysM66EZJM3A28tPnL2jNDgC  (mi + key1)


## Build aggregate sender identity (multiman)

In [11]:
agg_priv_sender = aggregate_privkey([cc_priv(priv_apo), cc_priv(priv_key1)])
agg_pub_sender  = aggregate_pubkey([pub_apo, pub_key1])
assert agg_priv_sender.public_key.format() == agg_pub_sender.format()
print(f'multiman aggregate pubkey: {agg_pub_sender.format().hex()}')
agg_pub_recipient = aggregate_pubkey([pub_test1, pub_test2, pub_test3])  # different multisig

multiman aggregate pubkey: 038eaa592590f4eb65c1e099ba6849deb4cde3525794ff311fc10ce7c2ab17095a


## Build inner + ECIES wrap

In [12]:
inner_h, inner_b = build_text_quipu('Del multiman 2-of-2',
                                    'Mensaje del 2-of-2 (mi+key1), tres sobres: a nosotros, a test1, al multisig de test1+test2+test3.',
                                    tone=TONE_ORDINARY)
recipients = [agg_pub_sender, pub_test1, agg_pub_recipient]
labels = ['sender self (multiman aggregate = mi+key1)',
          'single-key recipient (test1)',
          'DIFFERENT multikey aggregate (test_multisig3 = test1+test2+test3)']

from coincurve.utils import get_valid_secret
import ecies as _ecies
from encrypted import _shared_key, _frame_inner, MAGIC, TYPE_ENCRYPTED, SUB_ECIES, ECIES_BROADCAST, TONE_ORDINARY
session_key = get_valid_secret()
header = MAGIC + bytes([TYPE_ENCRYPTED, TONE_ORDINARY, SUB_ECIES, ECIES_BROADCAST])
header += b'|del multiman|'
envelopes = b''
for pub in recipients:
    envelopes += _ecies.sym_encrypt(_shared_key(agg_priv_sender, pub), session_key)
framed = _frame_inner(inner_h, inner_b)
ciphertext = _ecies.sym_encrypt(session_key, framed)
outer_h = header
outer_b = bytes([len(recipients)]) + envelopes + ciphertext
print(f'outer ({len(outer_h)+len(outer_b)} B); session key hex: {session_key.hex()}')
SESSION_KEY_PATH = os.path.join(INSCRIPTIONS_READY, 'ecies_multisig_session.bin')
with open(SESSION_KEY_PATH, 'wb') as f: f.write(session_key)
print(f'session key saved to {SESSION_KEY_PATH}')

outer (373 B); session key hex: 8b8a86c70600c23dceb8014b8f183dbe241ac7c7ce2612328106de188726cc99
session key saved to /Users/anthonyschultz/Desktop/Colegio_Invisible/inscriptions_ready/ecies_multisig_session.bin


## Inscribe — multisig diamond from multiman

In [13]:
N_BODY_STRANDS = 4
chunk = len(outer_b) // N_BODY_STRANDS
extra = len(outer_b) %  N_BODY_STRANDS
body_parts, i = [], 0
for k in range(N_BODY_STRANDS):
    sz = chunk + (1 if k < extra else 0)
    body_parts.append(outer_b[i:i+sz]); i += sz
strand_payloads = [outer_h] + body_parts
print(f'{len(strand_payloads)} strands; sizes: {[len(p) for p in strand_payloads]}')

5 strands; sizes: [22, 88, 88, 88, 87]


In [14]:
utxos = ct.rpc_request('listunspent', [0, 9999999, [addr_multiman]])
seed_inputs = [{'output': f"{u['txid']}:{u['vout']}", 'value': int(round(u['amount']*1e8))} for u in utxos]
total = sum(s['value'] for s in seed_inputs)
print(f'multiman UTXOs: {len(seed_inputs)}, total {total/1e8:.4f} DOGE')

multiman UTXOs: 1, total 38.8500 DOGE


### Phase I — multisig root tx with scaled fee

In [15]:
# Order must match the pubkey order in the redeem script
# multiman labels are ['key1_prv.enc', 'mi_prv.enc'] — derive prvkeys in that exact order
_label_to_priv = {'mi_prv.enc': priv_apo, 'key1_prv.enc': priv_key1}
prvkeys = [_label_to_priv[lbl].to_hex()[2:] for lbl in multiman['labels']]
for s in seed_inputs:
    s['script'] = redeem_multiman_hex
n = len(strand_payloads)

def sign_multisig(tx, inputs_for_sigs):
    sigs_per_input = [[] for _ in range(len(inputs_for_sigs))]
    for pk in prvkeys:
        for vin_i in range(len(inputs_for_sigs)):
            sigs_per_input[vin_i].append(doge.multisign(tx, vin_i, redeem_multiman_hex, pk))
    for vin_i, sigs in enumerate(sigs_per_input):
        tx = doge.apply_multisignatures(tx, vin_i, redeem_multiman_hex, sigs)
    return tx

# Draft pass with TIP_MULTI floor
draft_per = (total - TIP_MULTI) // n
draft_seeds = [draft_per] * n
draft = doge.mktx(seed_inputs, [{'value': s, 'address': addr_multiman} for s in draft_seeds])
draft = sign_multisig(draft, seed_inputs)
draft_hex = cryptos.serialize(draft)
root_fee = scaled_fee(draft_hex, TIP_MULTI)
print(f'multisig root draft size: {len(draft_hex)//2} B  ->  scaled fee: {root_fee/1e8:.4f} DOGE')

# Real pass
per = (total - root_fee) // n
remainder = (total - root_fee) - per * n
strand_seeds = [per] * n
strand_seeds[0] += remainder
root_outputs = [{'value': s, 'address': addr_multiman} for s in strand_seeds]
root_tx = doge.mktx(seed_inputs, root_outputs)
root_tx = sign_multisig(root_tx, seed_inputs)
root_hex = cryptos.serialize(root_tx)
root_txid = _txid_of_serial(root_hex)
assert ct.rpc_request('sendrawtransaction', [root_hex]) == root_txid
print(f'root_txid: {root_txid}')

multisig root draft size: 495 B  ->  scaled fee: 0.1000 DOGE
root_txid: 43ea13fd8323f6326052668770d76bd17795ae6b89aa0ae2090a5408a520c17e


In [16]:
# Wait for root to confirm
print(f'waiting for root to confirm... ({root_txid[:16]}…)')
start_h = ct.rpc_request('getblockcount')
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        info = ct.rpc_request('getrawtransaction', [root_txid, 1])
        confs = info.get('confirmations', 0)
        print(f'  {time.strftime("%H:%M:%S")}  block {h}  root confs: {confs}')
        if confs >= 1:
            print(f'✓ root confirmed in block {info.get("blockhash","?")}')
            break
        start_h = h
    time.sleep(15)

waiting for root to confirm... (43ea13fd8323f632…)
  10:19:22  block 6214444  root confs: 0
  10:20:22  block 6214445  root confs: 0
  10:21:37  block 6214446  root confs: 1
✓ root confirmed in block e753efcbf82757e99abb76eb92b8495a5b253596a6ee556d84efefc83ff93ea6


### Phase II — CadenaMultiAtom strand chains (TIP_MULTI per knot)

In [17]:
strands = []
for si, payload in enumerate(strand_payloads):
    cad = ct.CadenaMultiAtom(prvkeys=prvkeys, data=payload,
                              utxo_dct={'output': f'{root_txid}:{si}', 'value': strand_seeds[si]},
                              tip=TIP_MULTI)
    cad.precompute()
    strands.append(cad)
    print(f'  strand {si}: {len(cad.txns)} knots, terminus {cad.txn_ids[-1]}')
for si, cad in enumerate(strands):
    for hex_tx, txid in zip(cad.txns, cad.txn_ids):
        assert ct.rpc_request('sendrawtransaction', [hex_tx]) == txid
    print(f'  strand {si} broadcast')

  strand 0: 1 knots, terminus 2917e4f1f2ed96a87cf5796a79222b8e1214acdb132451dc3725282af872ed3f
  strand 1: 2 knots, terminus a26e5e1440e23880bdc0790b97b2587b3b140dec67bf57d35ce1e95c1ef57077
  strand 2: 2 knots, terminus 85e00451c3f7acdfe3e8f85cfdf0b4a8603fbc33b777dc2c1ca821817894229f
  strand 3: 2 knots, terminus 5d79a4c992daf1fc82df9a5868128ded8141b6d8381ea3d70214bf339a4b1d84
  strand 4: 2 knots, terminus 297eb14e4ffa9f2170fe895744f370a49d4e225056d8c91439b6c212a547572b
  strand 0 broadcast
  strand 1 broadcast
  strand 2 broadcast
  strand 3 broadcast
  strand 4 broadcast


In [18]:
# Wait for all strand termini to confirm
print('waiting for strand termini to confirm...')
start_h = ct.rpc_request('getblockcount')
termini = [c.txn_ids[-1] for c in strands]
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        confs = [ct.rpc_request('getrawtransaction', [t, 1]).get('confirmations', 0) for t in termini]
        print(f'  block {h}  ' + '  '.join(f's{i}:{c}' for i,c in enumerate(confs)))
        if all(c >= 1 for c in confs):
            print('✓ all strand termini confirmed')
            break
        start_h = h
    time.sleep(15)

waiting for strand termini to confirm...
  block 6214450  s0:2  s1:2  s2:2  s3:2  s4:2
✓ all strand termini confirmed


### Phase III — multisig join tx with scaled fee

In [19]:
join_inputs = [{'output': f'{c.txn_ids[-1]}:0',
                'value': strand_seeds[si] - TIP_MULTI * len(c.txns),
                'script': redeem_multiman_hex}
               for si, c in enumerate(strands)]
join_total = sum(i['value'] for i in join_inputs)

# Draft pass
draft = doge.mktx(join_inputs, [{'value': join_total - TIP_MULTI, 'address': addr_multiman}])
draft = sign_multisig(draft, join_inputs)
draft_hex = cryptos.serialize(draft)
join_fee = scaled_fee(draft_hex, TIP_MULTI)
print(f'multisig join draft size: {len(draft_hex)//2} B  ->  scaled fee: {join_fee/1e8:.4f} DOGE')

# Real pass
join_tx = doge.mktx(join_inputs, [{'value': join_total - join_fee, 'address': addr_multiman}])
join_tx = sign_multisig(join_tx, join_inputs)
join_hex = cryptos.serialize(join_tx)
join_txid = _txid_of_serial(join_hex)
assert ct.rpc_request('sendrawtransaction', [join_hex]) == join_txid
print(f'join_txid: {join_txid}')

multisig join draft size: 1672 B  ->  scaled fee: 0.3344 DOGE
join_txid: 8eddfd373ac3dad368e699b257735392cade88c7d5d17e7bfc12066a0204f0ff


In [20]:
# Wait for join to confirm
print(f'waiting for join to confirm... ({join_txid[:16]}…)')
start_h = ct.rpc_request('getblockcount')
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        info = ct.rpc_request('getrawtransaction', [join_txid, 1])
        confs = info.get('confirmations', 0)
        print(f'  {time.strftime("%H:%M:%S")}  block {h}  join confs: {confs}')
        if confs >= 1:
            print(f'✓ join confirmed in block {info.get("blockhash","?")}')
            break
        start_h = h
    time.sleep(15)

waiting for join to confirm... (8eddfd373ac3dad3…)
  10:25:26  block 6214452  join confs: 0
  10:25:56  block 6214453  join confs: 1
✓ join confirmed in block 703585951cb9e0dce6d095cec0754937ee5878848c42c700fea44a27d7dfd0c3


## Read back

In [21]:
# Walk the diamond using our local spender map
spender_map = {}
for si, cad in enumerate(strands):
    spender_map[f'{root_txid}:{si}'] = cad.txn_ids[0]
    for ki in range(len(cad.txn_ids) - 1):
        spender_map[f'{cad.txn_ids[ki]}:0'] = cad.txn_ids[ki+1]
def walk(start):
    out, cur = '', start
    while True:
        n = spender_map.get(cur)
        if not n: return out
        raw = ct.rpc_request('getrawtransaction', [n, 1])
        op = next((ct.extract_op_return(v) for v in raw['vout'] if ct.extract_op_return(v)), None)
        if not op: return out
        out += op; cur = f'{n}:0'
rec_h = bytes.fromhex(walk(f'{root_txid}:0'))
rec_b = b''.join(bytes.fromhex(walk(f'{root_txid}:{si}')) for si in range(1, len(strands)))
assert rec_h == outer_h and rec_b == outer_b
print('✓ recovered byte-identical')

✓ recovered byte-identical


## Decode path 1 — sender self (multiman 2-of-2)

In [22]:
agg_priv_recovered = aggregate_privkey([cc_priv(priv_apo), cc_priv(priv_key1)])
parsed_self = read_encrypted_quipu(rec_h, rec_b,
                                     my_privkey=agg_priv_recovered,
                                     author_pubkey=agg_pub_sender)
assert parsed_self['inner_body'] == inner_b
print('✓ multiman self-decrypted (mi + key1 cooperated)')

✓ multiman self-decrypted (mi + key1 cooperated)


## Decode path 2 — single-key recipient (test1)

In [23]:
parsed_single = read_encrypted_quipu(rec_h, rec_b,
                                       my_privkey=cc_priv(priv_test1),
                                       author_pubkey=agg_pub_sender)
assert parsed_single['inner_body'] == inner_b
print('✓ test1 decrypted via envelope 1')

✓ test1 decrypted via envelope 1


## Decode path 3 — different multikey aggregate (test_multisig3)

In [24]:
agg_priv_recipient = aggregate_privkey([cc_priv(priv_test1), cc_priv(priv_test2), cc_priv(priv_test3)])
parsed_multi = read_encrypted_quipu(rec_h, rec_b,
                                      my_privkey=agg_priv_recipient,
                                      author_pubkey=agg_pub_sender)
assert parsed_multi['inner_body'] == inner_b
print('✓ test_multisig3 aggregate (DIFFERENT 3-of-3) decrypted via envelope 2')

✓ test_multisig3 aggregate (DIFFERENT 3-of-3) decrypted via envelope 2


In [26]:
# Recovered content from all three decode paths
# All three should yield byte-identical inner content.
for name, p in [('self (multiman 2-of-2)', parsed_self),
                ('single-key (test1)',     parsed_single),
                ('multi (test_multisig3)', parsed_multi)]:
    inner = read_text_quipu(p['inner_header'], p['inner_body'])
    print(f'via {name}:')
    print(f'  title: {inner["title"]!r}')
    print(f'  tone:  0x{inner["tone"]:02x}')
    print(f'  body:  {inner["body"]!r}')
    print()

via self (multiman 2-of-2):
  title: 'Del multiman 2-of-2'
  tone:  0x00
  body:  'Mensaje del 2-of-2 (mi+key1), tres sobres: a nosotros, a test1, al multisig de test1+test2+test3.'

via single-key (test1):
  title: 'Del multiman 2-of-2'
  tone:  0x00
  body:  'Mensaje del 2-of-2 (mi+key1), tres sobres: a nosotros, a test1, al multisig de test1+test2+test3.'

via multi (test_multisig3):
  title: 'Del multiman 2-of-2'
  tone:  0x00
  body:  'Mensaje del 2-of-2 (mi+key1), tres sobres: a nosotros, a test1, al multisig de test1+test2+test3.'



## Manifest

In [25]:
json.dump({'root_txid': root_txid, 'join_txid': join_txid,
           'session_key_path': SESSION_KEY_PATH,
           'sender_aggregate_pubkey': agg_pub_sender.format().hex(),
           'sender_address': addr_multiman, 'recipients': labels},
          open(os.path.join(INSCRIPTIONS_READY, 'ecies_multisig_manifest.json'), 'w'),
          indent=2)
print('manifest saved')

manifest saved
